In [1]:
from pyspark.sql import SparkSession
import os
import json
from pprint import pprint
warehouse_path = r"C:\iceberg-warehouse"

print(os.listdir(warehouse_path))

['db', 'demo']


In [16]:
spark = SparkSession.builder \
    .appName("IcebergLocal") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    ) \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    ) \
    .config(
        "spark.sql.catalog.local.warehouse",
        "file:///C:/iceberg-warehouse"
    ) \
    .getOrCreate()

In [17]:
spark.sql("""
DROP TABLE IF EXISTS local.demo.dist_test
""")

DataFrame[]

In [18]:
spark.sql("""
DROP TABLE IF EXISTS local.demo.dist_none
""")

spark.sql("""
CREATE TABLE local.demo.dist_none (
    id BIGINT,
    country STRING
)
USING iceberg
PARTITIONED BY (country)
TBLPROPERTIES (
  'write.distribution-mode'='none'
)
""")

DataFrame[]

In [19]:
data = []

for i in range(10000):
    data.append((i, "IN"))

for i in range(10000, 20000):
    data.append((i, "US"))

for i in range(20000, 30000):
    data.append((i, "UK"))

df = spark.createDataFrame(data, ["id", "country"])

In [20]:
print(df.rdd.getNumPartitions())


8


In [21]:
df = df.repartition(50)

In [22]:
print(df.rdd.getNumPartitions())

50


In [23]:
df.writeTo("local.demo.dist_none").append()

In [24]:
spark.sql("""
SELECT
    partition,
    count(*) as files
FROM local.demo.dist_none.files
GROUP BY partition
""").show(truncate=False)

+---------+-----+
|partition|files|
+---------+-----+
|{US}     |50   |
|{IN}     |50   |
|{UK}     |50   |
+---------+-----+



In [29]:
spark.sql("""
SELECT
    file_path,
    partition,
    record_count
FROM local.demo.dist_none.files
ORDER BY partition
""").show(100, truncate=False)

+------------------------------------------------------------------------------------------------------------------------+---------+------------+
|file_path                                                                                                               |partition|record_count|
+------------------------------------------------------------------------------------------------------------------------+---------+------------+
|file:/C:/iceberg-warehouse/demo/dist_none/data/country=IN/00000-94-0ca52fbc-ce10-4ea0-8316-b29c28599963-0-00001.parquet |{IN}     |200         |
|file:/C:/iceberg-warehouse/demo/dist_none/data/country=IN/00001-95-0ca52fbc-ce10-4ea0-8316-b29c28599963-0-00001.parquet |{IN}     |200         |
|file:/C:/iceberg-warehouse/demo/dist_none/data/country=IN/00002-96-0ca52fbc-ce10-4ea0-8316-b29c28599963-0-00001.parquet |{IN}     |197         |
|file:/C:/iceberg-warehouse/demo/dist_none/data/country=IN/00003-97-0ca52fbc-ce10-4ea0-8316-b29c28599963-0-00001.parquet |{I

In [25]:
spark.sql("""
DROP TABLE IF EXISTS local.demo.dist_hash
""")

spark.sql("""
CREATE TABLE local.demo.dist_hash (
    id BIGINT,
    country STRING
)
USING iceberg
PARTITIONED BY (country)
TBLPROPERTIES (
  'write.distribution-mode'='hash'
)
""")

DataFrame[]

In [26]:
df.writeTo("local.demo.dist_hash").append()

In [27]:
spark.sql("""
SELECT
    partition,
    count(*) as files
FROM local.demo.dist_hash.files
GROUP BY partition
""").show(truncate=False)

+---------+-----+
|partition|files|
+---------+-----+
|{US}     |1    |
|{IN}     |1    |
|{UK}     |1    |
+---------+-----+



In [28]:
spark.sql("""
SELECT
    file_path,
    partition,
    record_count
FROM local.demo.dist_hash.files
ORDER BY partition
""").show(100, truncate=False)

+------------------------------------------------------------------------------------------------------------------------+---------+------------+
|file_path                                                                                                               |partition|record_count|
+------------------------------------------------------------------------------------------------------------------------+---------+------------+
|file:/C:/iceberg-warehouse/demo/dist_hash/data/country=IN/00000-204-932b0f0a-7eeb-4187-8555-59677df57ccc-0-00002.parquet|{IN}     |10000       |
|file:/C:/iceberg-warehouse/demo/dist_hash/data/country=UK/00000-204-932b0f0a-7eeb-4187-8555-59677df57ccc-0-00003.parquet|{UK}     |10000       |
|file:/C:/iceberg-warehouse/demo/dist_hash/data/country=US/00000-204-932b0f0a-7eeb-4187-8555-59677df57ccc-0-00001.parquet|{US}     |10000       |
+------------------------------------------------------------------------------------------------------------------------+--